In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

import estimator as standard_estimator


def get_git_commit(path: Path) -> str:
    """Return the current Git commit containing the given path."""
    return subprocess.check_output(
        ["git", "-C", str(path.resolve()), "rev-parse", "HEAD"],
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()


def load_module_from_file(module_name: str, file_path: Path):
    """Load a Python file under an explicit module name."""
    file_path = file_path.resolve()

    if not file_path.is_file():
        raise FileNotFoundError(f"Module file not found: {file_path}")

    spec = importlib.util.spec_from_file_location(
        module_name,
        file_path,
    )

    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot create import spec for: {file_path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module

    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(module_name, None)
        raise

    return module

# Standard lattice-estimator
standard_package_path = Path(standard_estimator.__file__).resolve()

standard_estimator_commit = get_git_commit(
    standard_package_path.parent
)


# Enhanced lattice-estimator
enhanced_repository = (
    Path.cwd()
    / "enhanced_lattice-estimator"
).resolve()

enhanced_package = (
    enhanced_repository
    / "estimator"
)

enhanced_spec = importlib.util.spec_from_file_location(
    "enhanced_estimator",
    enhanced_package / "__init__.py",
    submodule_search_locations=[str(enhanced_package)],
)

if enhanced_spec is None or enhanced_spec.loader is None:
    raise ImportError(
        f"Cannot load enhanced estimator from {enhanced_package}"
    )

enhanced_estimator = importlib.util.module_from_spec(
    enhanced_spec
)

sys.modules["enhanced_estimator"] = enhanced_estimator

try:
    enhanced_spec.loader.exec_module(enhanced_estimator)
except Exception:
    sys.modules.pop("enhanced_estimator", None)
    raise

enhanced_estimator_commit = get_git_commit(
    enhanced_repository
)


# PrimalMeetLWE estimator
primal_meet_lwe_repository = (
    Path.cwd()
    / "PrimalMeetLWE"
).resolve()

primal_meet_lwe_package = (
    primal_meet_lwe_repository
    / "estimator"
)

primal_meet_lwe_utils = load_module_from_file(
    "_primal_meet_lwe_utils",
    primal_meet_lwe_package / "utils.py",
)

_missing = object()
_previous_utils = sys.modules.get("utils", _missing)

sys.modules["utils"] = primal_meet_lwe_utils

try:
    primal_meet_lwe_estimator = load_module_from_file(
        "primal_meet_lwe_estimator",
        primal_meet_lwe_package / "estimator.py",
    )
finally:
    if _previous_utils is _missing:
        sys.modules.pop("utils", None)
    else:
        sys.modules["utils"] = _previous_utils

primal_meet_lwe_estimator_commit = get_git_commit(
    primal_meet_lwe_repository
)


print("standard estimator commit:      ", standard_estimator_commit)
print("enhanced estimator commit:      ", enhanced_estimator_commit)
print("PrimalMeetLWE estimator commit: ", primal_meet_lwe_estimator_commit)

standard estimator commit:       3e48ef421ec256afddb3e7d2249a77eab6e9ba12
enhanced estimator commit:       876b66173f4354a96ddafc0ce3a79767ec43c6d4
PrimalMeetLWE estimator commit:  61115115830c909e42758f2774606074bf98afb1


In [3]:
# END-512
print('# ' * 32)
print('standard lattice estimator')
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=512, q=257, Xs=standard_estimator.ND.SparseTernary(n=512, p=72, m=72), Xe=standard_estimator.ND.SparseTernary(n=512, p=72, m=72)), jobs=16)
print('- ' * 32)
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=512, n=512, q=257, Xs=standard_estimator.ND.Uniform(0, 1), Xe=standard_estimator.ND.Uniform(0, 3)), jobs=16)

print('# ' * 32)
print('enhanced lattice estimator from \'Careful with the Ring! Concrete Hardness Gaps Between LWE and MLWE\' CRYPTO 2026')
params = enhanced_estimator.LWE.Parameters(m=512, n=512, q=257, Xs=enhanced_estimator.ND.Uniform(0, 1), Xe=enhanced_estimator.ND.Uniform(0, 3))
# Evaluate complexity under enhanced guess-and-verify decoding attack
res = enhanced_estimator.LWE.primal_hybrid(params, mitm=False, babai=False, deg_ring=params.n, structure_leverage=True)  
print(f'egv:{res}')
print('- ' * 32)
# Evaluate complexity under enhanced Howgrave-Graham decoding attack
res = enhanced_estimator.LWE.primal_hybrid(params, mitm=True, babai=True, deg_ring=params.n, structure_leverage=True)
print(f'ehg:{res}')

print('# ' * 32)
print('primal meet lwe hybrid attack estimator from \'A Hybrid of Lattice-reduction and Meet-LWE via Near-Collision on Babai\'s Plane\' DCC 2026')
try:
    # NTRU
    param = [512, 257, 'gaussian', float(standard_estimator.ND.SparseTernary(n=512, p=72, m=72).stddev), 2*72, 512]
    _ = primal_meet_lwe_estimator.primal_may(param, t=2)
    print('- ' * 32)
    # RLWR
    param = [512, 257, 'gaussian', float(standard_estimator.ND.Uniform(0, 3).stddev), 256, 512]
    _ = primal_meet_lwe_estimator.primal_may(param, t=2)
except KeyError:
    print('too high concrete security to estimate')

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
standard lattice estimator
usvp                 :: rop: ≈2^157.2, red: ≈2^157.2, δ: 1.003634, β: 456, d: 872, tag: usvp
bdd                  :: rop: ≈2^152.7, red: ≈2^151.1, svp: ≈2^152.1, β: 434, η: 469, d: 857, tag: bdd
bdd_hybrid           :: rop: ≈2^152.7, red: ≈2^151.1, svp: ≈2^152.1, β: 434, η: 469, ζ: 0, |S|: 1, d: 870, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^191.0, red: ≈2^189.4, svp: ≈2^190.4, β: 413, η: 2, ζ: 260, |S|: ≈2^253.2, d: 600, prob: ≈2^-43.1, ↻: ≈2^45.3, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
Algorithm <estimator.lwe_bkw.CodedBKW object at 0x78118e522320> on LWEParameters(n=512, q=257, Xs=D(σ=0.50, μ=0.50), Xe=D(σ=1.12, μ=1.50), m=512, tag=None) failed with Amplifying for μ≠0 not implemented.
usvp                 :: rop: ≈2^170.7, red: ≈2^170.7, δ: 1.003375, β: 506, d: 825, tag: usvp
bdd                  :: rop: ≈2^165.9, red: ≈2^164.1, svp: ≈2

In [4]:
# END-1024
print('# ' * 32)
print('standard lattice estimator')
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=1024, q=257, Xs=standard_estimator.ND.SparseTernary(n=1024, p=96, m=96), Xe=standard_estimator.ND.SparseTernary(n=1024, p=96, m=96)), jobs=16)
print('- ' * 32)
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=1024, n=1024, q=257, Xs=standard_estimator.ND.Uniform(0, 1), Xe=standard_estimator.ND.SparseTernary(n=1024, p=floor(90*1024/257), m=floor(76*1024/257))), jobs=16)

print('# ' * 32)
print('enhanced lattice estimator from \'Careful with the Ring! Concrete Hardness Gaps Between LWE and MLWE\' CRYPTO 2026')
params = enhanced_estimator.LWE.Parameters(m=1024, n=1024, q=257, Xs=enhanced_estimator.ND.Uniform(0, 1), Xe=enhanced_estimator.ND.SparseTernary(n=1024, p=floor(90*1024/257), m=floor(76*1024/257)))
# Evaluate complexity under enhanced guess-and-verify decoding attack
res = enhanced_estimator.LWE.primal_hybrid(params, mitm=False, babai=False, deg_ring=params.n, structure_leverage=True)  
print(f'egv:\n{res}')
print('- ' * 32)
# Evaluate complexity under enhanced Howgrave-Graham decoding attack
res = enhanced_estimator.LWE.primal_hybrid(params, mitm=True, babai=True, deg_ring=params.n, structure_leverage=True)
print(f'ehg:\n{res}')

print('# ' * 32)
print('primal meet lwe hybrid attack estimator from \'A Hybrid of Lattice-reduction and Meet-LWE via Near-Collision on Babai\'s Plane\' DCC 2026')
try:
    # NTRU
    param = [1024, 257, 'gaussian', float(standard_estimator.ND.SparseTernary(n=1024, p=96, m=96).stddev), 2*96, 1024]
    _ = primal_meet_lwe_estimator.primal_may(param, t=2)
    print('- ' * 32)
    # RLWR
    param = [1024, 257, 'gaussian', float(standard_estimator.ND.SparseTernary(n=1024, p=floor(90*1024/257), m=floor(76*1024/257)).stddev), 512, 1024]
    _ = primal_meet_lwe_estimator.primal_may(param, t=2)
except KeyError:
    print('too high concrete security to estimate')

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
standard lattice estimator
usvp                 :: rop: ≈2^292.5, red: ≈2^292.5, δ: 1.002144, β: 938, d: 1621, tag: usvp
bdd                  :: rop: ≈2^287.1, red: ≈2^285.8, svp: ≈2^286.4, β: 914, η: 950, d: 1598, tag: bdd
bdd_hybrid           :: rop: ≈2^287.2, red: ≈2^285.8, svp: ≈2^286.4, β: 914, η: 950, ζ: 0, |S|: 1, d: 1613, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^310.0, red: ≈2^308.2, svp: ≈2^309.5, β: 735, η: 2, ζ: 621, |S|: ≈2^431.0, d: 933, prob: ≈2^-72.1, ↻: ≈2^74.3, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
Algorithm <estimator.lwe_bkw.CodedBKW object at 0x78118e522320> on LWEParameters(n=1024, q=257, Xs=D(σ=0.50, μ=0.50), Xe=T(p=358, m=302, n=1024), m=1024, tag=None) failed with Amplifying for μ≠0 not implemented.
usvp                 :: rop: ≈2^321.6, red: ≈2^321.6, δ: 1.001979, β: 1043, d: 1583, tag: usvp
bdd                  :: rop: ≈2^315.9, red: ≈2^3

In [16]:
# ML-KEM-512
_ = standard_estimator.LWE.estimate(standard_estimator.schemes.Kyber512, jobs=16)

bkw                  :: rop: ≈2^178.8, m: ≈2^166.8, mem: ≈2^167.8, b: 14, t1: 0, t2: 16, ℓ: 13, #cod: 448, #top: 0, #test: 64, tag: coded-bkw
usvp                 :: rop: ≈2^143.8, red: ≈2^143.8, δ: 1.003941, β: 406, d: 998, tag: usvp
bdd                  :: rop: ≈2^140.2, red: ≈2^139.1, svp: ≈2^139.3, β: 389, η: 422, d: 1005, tag: bdd
dual                 :: rop: ≈2^149.9, mem: ≈2^97.1, m: 512, β: 424, d: 1024, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^139.7, red: ≈2^139.5, guess: ≈2^135.9, β: 387, p: 5, ζ: 0, t: 50, β': 391, N: ≈2^81.1, m: 512


In [17]:
# ML-KEM-1024
_ = standard_estimator.LWE.estimate(standard_estimator.schemes.Kyber1024, jobs=16)

bkw                  :: rop: ≈2^315.0, m: ≈2^301.0, mem: ≈2^296.7, b: 25, t1: 0, t2: 18, ℓ: 24, #cod: 897, #top: 0, #test: 129, tag: coded-bkw
usvp                 :: rop: ≈2^275.1, red: ≈2^275.1, δ: 1.002262, β: 874, d: 1867, tag: usvp
bdd                  :: rop: ≈2^270.7, red: ≈2^269.8, svp: ≈2^269.6, β: 855, η: 889, d: 1867, tag: bdd
dual                 :: rop: ≈2^288.5, mem: ≈2^195.3, m: 930, β: 918, d: 1954, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^262.3, red: ≈2^261.9, guess: ≈2^260.3, β: 823, p: 4, ζ: 0, t: 120, β': 804, N: ≈2^165.9, m: 1024


In [5]:
# NEV-512 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=512, q=769, Xs=standard_estimator.ND.CenteredBinomial(1), Xe=standard_estimator.ND.CenteredBinomial(1)), jobs=16)
print('- ' * 32)
# NEV-512 RLWE
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=512, n=512, q=769, Xs=standard_estimator.ND.CenteredBinomial(1), Xe=standard_estimator.ND.SparseTernary(n=512, p=floor(1/6*512), m=floor(1/6*512))), jobs=16)

usvp                 :: rop: ≈2^149.1, red: ≈2^149.1, δ: 1.003811, β: 426, d: 934, tag: usvp
bdd                  :: rop: ≈2^145.1, red: ≈2^144.1, svp: ≈2^144.0, β: 408, η: 440, d: 912, tag: bdd
bdd_hybrid           :: rop: ≈2^145.1, red: ≈2^144.2, svp: ≈2^144.0, β: 408, η: 440, ζ: 0, |S|: 1, d: 932, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^207.0, red: ≈2^204.5, svp: ≈2^206.7, β: 427, η: 2, ζ: 184, |S|: ≈2^263.7, d: 758, prob: ≈2^-53.5, ↻: ≈2^55.7, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
arora-gb             :: rop: ≈2^794.4, dreg: 92, mem: ≈2^588.3, t: 1, m: ≈2^215.1, tag: arora-gb, ↻: ≈2^206.1, ζ: 236, |S|: 1, prop: ≈2^-203.9
bkw                  :: rop: ≈2^148.6, m: ≈2^136.9, mem: ≈2^137.9, b: 14, t1: 0, t2: 13, ℓ: 13, #cod: 424, #top: 0, #test: 90, tag: coded-bkw
usvp                 :: rop: ≈2^144.9, red: ≈2^144.9, δ: 1.003908, β: 411, d: 889, tag: usvp
bdd                  :: rop: ≈2^141.0, red: ≈2^139.9, svp: ≈2^140.1, β: 

In [6]:
# DAWN-alpha-512 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=512, q=769, Xs=standard_estimator.ND.SparseTernary(n=512, p=160, m=160), Xe=standard_estimator.ND.SparseTernary(n=512, p=64, m=64)), jobs=16)
print('- ' * 32)
# DAWN-alpha-512 RLWE
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=512, n=512, q=769, Xs=standard_estimator.ND.SparseTernary(n=512, p=160, m=160), Xe=standard_estimator.ND.SparseTernary(n=512, p=96, m=96)), jobs=16)

Algorithm functools.partial(<estimator.ntru_primal.PrimalDSD object at 0x78118e57ce20>, red_cost_model=<estimator.reduction.MATZOV object at 0x78118e4ed330>, red_shape_model=<function GSA at 0x78118e501480>) on NTRUParameters(n=512, q=769, Xs=T(p=64, m=64, n=512), Xe=T(p=160, m=160, n=512), m=512, tag=None, ntru_type='matrix') failed with Dense sublattice attack not supported for Xs != Xe
usvp                 :: rop: ≈2^143.5, red: ≈2^143.5, δ: 1.003941, β: 406, d: 884, tag: usvp
bdd                  :: rop: ≈2^139.4, red: ≈2^137.9, svp: ≈2^138.7, β: 386, η: 421, d: 873, tag: bdd
bdd_hybrid           :: rop: ≈2^139.4, red: ≈2^138.0, svp: ≈2^138.7, β: 386, η: 421, ζ: 0, |S|: 1, d: 882, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^169.3, red: ≈2^168.0, svp: ≈2^168.5, β: 355, η: 2, ζ: 251, |S|: ≈2^220.9, d: 612, prob: ≈2^-37.3, ↻: ≈2^39.5, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
arora-gb             :: rop: ≈2^833.3, dreg: 92, mem: ≈2^5

In [7]:
# DAWN-alpha-1024 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=1024, q=769, Xs=standard_estimator.ND.SparseTernary(n=1024, p=256, m=256), Xe=standard_estimator.ND.SparseTernary(n=1024, p=96, m=96)), jobs=16)
print('- ' * 32)
# DAWN-alpha-1024 RLWE
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=1024, n=1024, q=769, Xs=standard_estimator.ND.SparseTernary(n=1024, p=256, m=256), Xe=standard_estimator.ND.SparseTernary(n=1024, p=192, m=192)), jobs=16)

Algorithm functools.partial(<estimator.ntru_primal.PrimalDSD object at 0x78118e57ce20>, red_cost_model=<estimator.reduction.MATZOV object at 0x78118e4ed330>, red_shape_model=<function GSA at 0x78118e501480>) on NTRUParameters(n=1024, q=769, Xs=T(p=96, m=96, n=1024), Xe=T(p=256, m=256, n=1024), m=1024, tag=None, ntru_type='matrix') failed with Dense sublattice attack not supported for Xs != Xe
usvp                 :: rop: ≈2^274.7, red: ≈2^274.7, δ: 1.002262, β: 874, d: 1636, tag: usvp
bdd                  :: rop: ≈2^269.7, red: ≈2^268.6, svp: ≈2^268.8, β: 852, η: 887, d: 1632, tag: bdd
bdd_hybrid           :: rop: ≈2^269.7, red: ≈2^268.6, svp: ≈2^268.8, β: 852, η: 887, ζ: 0, |S|: 1, d: 1655, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^292.7, red: ≈2^291.2, svp: ≈2^292.1, β: 690, η: 2, ζ: 589, |S|: ≈2^406.0, d: 998, prob: ≈2^-67.0, ↻: ≈2^69.2, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
arora-gb             :: rop: ≈2^inf, dreg: 153, mem

In [8]:
# DAWN-beta-512 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=512, q=257, Xs=standard_estimator.ND.SparseTernary(n=512, p=64, m=64), Xe=standard_estimator.ND.SparseTernary(n=512, p=32, m=32)), jobs=16)
print('- ' * 32)
# DAWN-beta-512 RLWE
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=512, n=512, q=257, Xs=standard_estimator.ND.SparseTernary(n=512, p=64, m=64), Xe=standard_estimator.ND.SparseTernary(n=512, p=48, m=48)), jobs=16)

Algorithm functools.partial(<estimator.ntru_primal.PrimalDSD object at 0x78118e57ce20>, red_cost_model=<estimator.reduction.MATZOV object at 0x78118e4ed330>, red_shape_model=<function GSA at 0x78118e501480>) on NTRUParameters(n=512, q=257, Xs=T(p=32, m=32, n=512), Xe=T(p=64, m=64, n=512), m=512, tag=None, ntru_type='matrix') failed with Dense sublattice attack not supported for Xs != Xe
usvp                 :: rop: ≈2^145.4, red: ≈2^145.4, δ: 1.003888, β: 414, d: 805, tag: usvp
bdd                  :: rop: ≈2^141.0, red: ≈2^139.9, svp: ≈2^140.1, β: 394, η: 426, d: 799, tag: bdd
bdd_hybrid           :: rop: ≈2^138.2, red: ≈2^137.1, svp: ≈2^137.3, β: 286, η: 288, ζ: 133, |S|: 267, d: 629, prob: ≈2^-25.2, ↻: ≈2^27.4, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^133.5, red: ≈2^131.6, svp: ≈2^133.0, β: 238, η: 2, ζ: 337, |S|: ≈2^159.1, d: 403, prob: ≈2^-34.0, ↻: ≈2^36.2, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
arora-gb             :: rop: ≈2^453.5, dreg: 

In [9]:
# DAWN-beta-1024 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=1024, q=257, Xs=standard_estimator.ND.SparseTernary(n=1024, p=96, m=96), Xe=standard_estimator.ND.SparseTernary(n=1024, p=64, m=64)), jobs=16)
print('- ' * 32)
# DAWN-beta-1024 RLWE
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=1024, n=1024, q=257, Xs=standard_estimator.ND.SparseTernary(n=1024, p=96, m=96), Xe=standard_estimator.ND.SparseTernary(n=1024, p=96, m=96)), jobs=16)

Algorithm functools.partial(<estimator.ntru_primal.PrimalDSD object at 0x78118e57ce20>, red_cost_model=<estimator.reduction.MATZOV object at 0x78118e4ed330>, red_shape_model=<function GSA at 0x78118e501480>) on NTRUParameters(n=1024, q=257, Xs=T(p=64, m=64, n=1024), Xe=T(p=96, m=96, n=1024), m=1024, tag=None, ntru_type='matrix') failed with Dense sublattice attack not supported for Xs != Xe
usvp                 :: rop: ≈2^281.8, red: ≈2^281.8, δ: 1.002212, β: 900, d: 1550, tag: usvp
bdd                  :: rop: ≈2^276.5, red: ≈2^275.1, svp: ≈2^275.8, β: 876, η: 912, d: 1536, tag: bdd
bdd_hybrid           :: rop: ≈2^266.0, red: ≈2^264.6, svp: ≈2^265.3, β: 614, η: 616, ζ: 291, |S|: 583, d: 1157, prob: ≈2^-60.8, ↻: ≈2^63.0, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^252.9, red: ≈2^251.0, svp: ≈2^252.4, β: 557, η: 2, ζ: 696, |S|: ≈2^332.9, d: 747, prob: ≈2^-64.7, ↻: ≈2^66.9, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
arora-gb             :: rop: ≈2^966.8,

In [10]:
# BAT-512 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=512, q=257, Xs=standard_estimator.ND.DiscreteGaussian(0.596), Xe=standard_estimator.ND.DiscreteGaussian(0.596)), jobs=16)
print('- ' * 32)
# BAT-512 RLWR
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=512, n=512, q=257, Xs=standard_estimator.ND.Uniform(0, 1), Xe=standard_estimator.ND.Uniform(0, 1)), jobs=16)

usvp                 :: rop: ≈2^162.7, red: ≈2^162.7, δ: 1.003525, β: 476, d: 881, tag: usvp
bdd                  :: rop: ≈2^158.0, red: ≈2^156.6, svp: ≈2^157.4, β: 454, η: 488, d: 873, tag: bdd
bdd_hybrid           :: rop: ≈2^158.0, red: ≈2^156.7, svp: ≈2^157.4, β: 454, η: 488, ζ: 0, |S|: 1, d: 884, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^342.9, red: ≈2^342.9, svp: ≈2^199.5, β: 477, η: 2, ζ: 0, |S|: 1, d: 900, prob: ≈2^-177.7, ↻: ≈2^179.9, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
Algorithm <estimator.lwe_bkw.CodedBKW object at 0x78118e522320> on LWEParameters(n=512, q=257, Xs=D(σ=0.50, μ=0.50), Xe=D(σ=0.50, μ=0.50), m=512, tag=None) failed with Amplifying for μ≠0 not implemented.
arora-gb             :: rop: ≈2^445.0, dreg: 23, mem: ≈2^232.0, t: 0, m: ≈2^222.0, tag: arora-gb, ↻: ≈2^213.0, ζ: 213
usvp                 :: rop: ≈2^154.6, red: ≈2^154.6, δ: 1.003685, β: 447, d: 853, tag: usvp
bdd                  :: rop: ≈2^150.4, red

In [11]:
# BAT-1024 NTRU
_ = standard_estimator.NTRU.estimate(standard_estimator.NTRU.Parameters(n=1024, q=769, Xs=standard_estimator.ND.DiscreteGaussian(0.659), Xe=standard_estimator.ND.DiscreteGaussian(0.659)), jobs=16)
print('- ' * 32)
# BAT-1024 RLWR
_ = standard_estimator.LWE.estimate(standard_estimator.LWE.Parameters(m=1024, n=1024, q=769, Xs=standard_estimator.ND.Uniform(0, 1), Xe=standard_estimator.ND.Uniform(0, 3)), jobs=16)

usvp                 :: rop: ≈2^291.6, red: ≈2^291.6, δ: 1.002151, β: 934, d: 1729, tag: usvp
bdd                  :: rop: ≈2^286.4, red: ≈2^285.5, svp: ≈2^285.3, β: 912, η: 946, d: 1752, tag: bdd
bdd_hybrid           :: rop: ≈2^286.4, red: ≈2^285.5, svp: ≈2^285.3, β: 912, η: 946, ζ: 0, |S|: 1, d: 1764, prob: 1.0, ↻: 1, tag: hybrid
bdd_mitm_hybrid      :: rop: ≈2^735.2, red: ≈2^735.2, svp: ≈2^464.9, β: 935, η: 2, ζ: 0, |S|: 1, d: 1781, prob: ≈2^-441.1, ↻: ≈2^443.3, tag: hybrid
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
Algorithm <estimator.lwe_bkw.CodedBKW object at 0x78118e522320> on LWEParameters(n=1024, q=769, Xs=D(σ=0.50, μ=0.50), Xe=D(σ=1.12, μ=1.50), m=1024, tag=None) failed with Amplifying for μ≠0 not implemented.
usvp                 :: rop: ≈2^295.6, red: ≈2^295.6, δ: 1.002126, β: 949, d: 1633, tag: usvp
bdd                  :: rop: ≈2^290.3, red: ≈2^288.7, svp: ≈2^289.8, β: 924, η: 961, d: 1652, tag: bdd
dual                 :: rop: ≈2^312.9, mem: ≈2^212